# Distributed Data Parallel (DDP) Tutorial

## Overview

This tutorial covers PyTorch's DistributedDataParallel (DDP), the most widely used approach for distributed training.

### Learning Objectives
- Understand DDP architecture and communication patterns
- Implement DDP training from scratch
- Master gradient synchronization with AllReduce
- Optimize DDP performance

### Prerequisites
- PyTorch basics (nn.Module, autograd)
- Multi-GPU hardware or simulation environment

### References
- Li et al., "PyTorch Distributed: Experiences on Accelerating Data Parallel Training", VLDB 2020
- [PyTorch DDP Tutorial](https://pytorch.org/tutorials/intermediate/ddp_tutorial.html)

## 1. Mathematical Foundation

### 1.1 Data Parallelism Principle

In data parallelism, we split a mini-batch $B$ across $N$ workers:

$$B = B_1 \cup B_2 \cup ... \cup B_N$$

Each worker $i$ computes local gradients:

$$g_i = \frac{1}{|B_i|} \sum_{x \in B_i} \nabla_\theta L(x, \theta)$$

The global gradient is the average:

$$g = \frac{1}{N} \sum_{i=1}^{N} g_i$$

### 1.2 AllReduce Communication

DDP uses Ring-AllReduce for gradient synchronization:

**Communication Complexity:**
$$T_{\text{AllReduce}} = 2(N-1) \cdot \frac{M}{N} \cdot (\alpha + \beta \cdot \frac{M}{N})$$

Where:
- $N$ = number of workers
- $M$ = message size (gradient size)
- $\alpha$ = latency per message
- $\beta$ = bandwidth cost per byte

**Bandwidth Optimal:** Ring-AllReduce achieves:
$$\text{Bandwidth} = \frac{2(N-1)}{N} \cdot M \approx 2M \text{ (for large N)}$$

## 2. Environment Setup

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler
import torch.multiprocessing as mp

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
print(f"NCCL available: {torch.distributed.is_nccl_available()}")

## 3. DDP Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                    DDP Training Flow                        │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│   GPU 0              GPU 1              GPU 2              │
│   ┌─────┐            ┌─────┐            ┌─────┐            │
│   │Model│            │Model│            │Model│            │
│   │Copy │            │Copy │            │Copy │            │
│   └──┬──┘            └──┬──┘            └──┬──┘            │
│      │                  │                  │                │
│   ┌──▼──┐            ┌──▼──┐            ┌──▼──┐            │
│   │Data │            │Data │            │Data │            │
│   │Shard│            │Shard│            │Shard│            │
│   └──┬──┘            └──┬──┘            └──┬──┘            │
│      │                  │                  │                │
│   ┌──▼──┐            ┌──▼──┐            ┌──▼──┐            │
│   │Local│            │Local│            │Local│            │
│   │Grad │            │Grad │            │Grad │            │
│   └──┬──┘            └──┬──┘            └──┬──┘            │
│      │                  │                  │                │
│      └──────────────────┼──────────────────┘                │
│                         │                                   │
│                  ┌──────▼──────┐                            │
│                  │  AllReduce  │                            │
│                  │  (Ring)     │                            │
│                  └──────┬──────┘                            │
│                         │                                   │
│      ┌──────────────────┼──────────────────┐                │
│      │                  │                  │                │
│   ┌──▼──┐            ┌──▼──┐            ┌──▼──┐            │
│   │Sync │            │Sync │            │Sync │            │
│   │Grad │            │Grad │            │Grad │            │
│   └─────┘            └─────┘            └─────┘            │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

## 4. Basic DDP Implementation

In [ ]:
class SimpleModel(nn.Module):
    """Simple MLP for demonstration."""
    
    def __init__(self, input_dim: int = 784, hidden_dim: int = 256, output_dim: int = 10):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        return self.layers(x.view(x.size(0), -1))

In [ ]:
def setup_distributed(rank: int, world_size: int):
    """Initialize distributed process group.
    
    Args:
        rank: Process rank (0 to world_size-1)
        world_size: Total number of processes
    """
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    
    # Initialize process group with NCCL backend (optimal for GPU)
    dist.init_process_group(
        backend='nccl',  # Use 'gloo' for CPU
        rank=rank,
        world_size=world_size
    )
    
    # Set device for this process
    torch.cuda.set_device(rank)


def cleanup_distributed():
    """Clean up distributed process group."""
    dist.destroy_process_group()

In [ ]:
def create_ddp_model(model: nn.Module, rank: int) -> DDP:
    """Wrap model with DistributedDataParallel.
    
    Args:
        model: PyTorch model
        rank: Process rank
        
    Returns:
        DDP-wrapped model
    """
    model = model.to(rank)
    
    ddp_model = DDP(
        model,
        device_ids=[rank],
        output_device=rank,
        find_unused_parameters=False  # Set True if model has unused params
    )
    
    return ddp_model

## 5. Training Loop with DDP

In [ ]:
def train_epoch(model, dataloader, optimizer, criterion, rank):
    """Train for one epoch.
    
    Args:
        model: DDP-wrapped model
        dataloader: DataLoader with DistributedSampler
        optimizer: Optimizer
        criterion: Loss function
        rank: Process rank
        
    Returns:
        Average loss for the epoch
    """
    model.train()
    total_loss = 0.0
    num_batches = 0
    
    for batch_idx, (data, target) in enumerate(dataloader):
        data, target = data.to(rank), target.to(rank)
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        
        # Backward pass - gradients are automatically synchronized
        loss.backward()
        
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / num_batches

In [ ]:
def ddp_training_worker(rank: int, world_size: int, epochs: int = 5):
    """Worker function for DDP training.
    
    Args:
        rank: Process rank
        world_size: Total number of processes
        epochs: Number of training epochs
    """
    setup_distributed(rank, world_size)
    
    # Create model and wrap with DDP
    model = SimpleModel()
    ddp_model = create_ddp_model(model, rank)
    
    # Create synthetic dataset
    dataset = torch.utils.data.TensorDataset(
        torch.randn(1000, 784),
        torch.randint(0, 10, (1000,))
    )
    
    # DistributedSampler ensures each process gets different data
    sampler = DistributedSampler(
        dataset,
        num_replicas=world_size,
        rank=rank,
        shuffle=True
    )
    
    dataloader = DataLoader(
        dataset,
        batch_size=32,
        sampler=sampler,
        num_workers=0,
        pin_memory=True
    )
    
    optimizer = optim.Adam(ddp_model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        # Important: set epoch for proper shuffling
        sampler.set_epoch(epoch)
        
        loss = train_epoch(ddp_model, dataloader, optimizer, criterion, rank)
        
        if rank == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}")
    
    cleanup_distributed()

## 6. Launch DDP Training

DDP training can be launched in multiple ways:

### Method 1: torch.multiprocessing.spawn

In [ ]:
def run_ddp_spawn(world_size: int = 2):
    """Launch DDP training using mp.spawn."""
    mp.spawn(
        ddp_training_worker,
        args=(world_size,),
        nprocs=world_size,
        join=True
    )

# Uncomment to run (requires multiple GPUs)
# run_ddp_spawn(world_size=2)

### Method 2: torchrun (Recommended for production)

```bash
# Single node, multiple GPUs
torchrun --nproc_per_node=4 train.py

# Multiple nodes
torchrun --nnodes=2 --nproc_per_node=4 \
         --rdzv_id=100 --rdzv_backend=c10d \
         --rdzv_endpoint=$MASTER_ADDR:29400 \
         train.py
```

## 7. Gradient Bucket and Communication Optimization

DDP uses gradient bucketing to overlap communication with computation.

In [ ]:
def create_optimized_ddp_model(model: nn.Module, rank: int) -> DDP:
    """Create DDP model with optimized settings.
    
    Key optimizations:
    - bucket_cap_mb: Controls gradient bucket size
    - gradient_as_bucket_view: Reduces memory copy
    - static_graph: Enables additional optimizations for static models
    """
    model = model.to(rank)
    
    ddp_model = DDP(
        model,
        device_ids=[rank],
        output_device=rank,
        bucket_cap_mb=25,  # Default is 25MB
        gradient_as_bucket_view=True,  # Memory optimization
        static_graph=False,  # Set True for static computation graphs
    )
    
    return ddp_model

### Bucket Size Analysis

```
Bucket Size vs Communication Efficiency

Small buckets (1-5 MB):
├── Pros: Better overlap with computation
├── Cons: Higher latency overhead
└── Best for: Small models, high-latency networks

Large buckets (25-50 MB):
├── Pros: Better bandwidth utilization
├── Cons: Less overlap opportunity
└── Best for: Large models, high-bandwidth networks
```

## 8. Performance Profiling

In [ ]:
def profile_ddp_communication():
    """Profile DDP communication overhead."""
    import time
    
    if not dist.is_initialized():
        print("Distributed not initialized. Showing simulation.")
        return
    
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    
    # Test different tensor sizes
    sizes = [1_000, 10_000, 100_000, 1_000_000, 10_000_000]
    
    for size in sizes:
        tensor = torch.randn(size, device=f'cuda:{rank}')
        
        # Warmup
        dist.all_reduce(tensor.clone())
        torch.cuda.synchronize()
        
        # Measure
        start = time.perf_counter()
        for _ in range(10):
            dist.all_reduce(tensor.clone())
        torch.cuda.synchronize()
        elapsed = (time.perf_counter() - start) / 10
        
        bandwidth = (size * 4 * 2 * (world_size - 1) / world_size) / elapsed / 1e9
        
        if rank == 0:
            print(f"Size: {size:>10}, Time: {elapsed*1000:.2f}ms, BW: {bandwidth:.2f} GB/s")

## 9. Common Issues and Solutions

### Issue 1: Gradient Mismatch
```python
# Problem: Different random operations on different ranks
# Solution: Set same seed or use DistributedSampler
torch.manual_seed(42 + rank)  # Different seed per rank for data
```

### Issue 2: Unused Parameters
```python
# Problem: Model has parameters not used in forward pass
# Solution: Set find_unused_parameters=True
ddp_model = DDP(model, find_unused_parameters=True)
```

### Issue 3: Deadlock
```python
# Problem: Different control flow on different ranks
# Solution: Ensure all ranks execute same operations
if rank == 0:
    # This causes deadlock!
    output = model(data)
```

## 10. Scaling Efficiency Analysis

### Linear Scaling Rule

When scaling from 1 GPU to N GPUs:
- Effective batch size: $B_{\text{eff}} = N \times B_{\text{local}}$
- Learning rate scaling: $\eta_{\text{new}} = N \times \eta_{\text{base}}$ (linear scaling)
- Or use warmup: gradually increase LR over first few epochs

### Scaling Efficiency

$$\text{Efficiency} = \frac{T_1}{N \times T_N}$$

Where:
- $T_1$ = time for 1 GPU
- $T_N$ = time for N GPUs
- Ideal efficiency = 100%

In [ ]:
def calculate_scaling_efficiency(t1: float, tn: float, n: int) -> float:
    """Calculate scaling efficiency.
    
    Args:
        t1: Time for single GPU
        tn: Time for N GPUs
        n: Number of GPUs
        
    Returns:
        Scaling efficiency (0-1)
    """
    return t1 / (n * tn)

# Example: 100s on 1 GPU, 30s on 4 GPUs
efficiency = calculate_scaling_efficiency(100, 30, 4)
print(f"Scaling efficiency: {efficiency:.1%}")

## 11. Summary

### Key Takeaways

1. **DDP Architecture**: Each GPU has a full model copy, data is sharded
2. **AllReduce**: Ring-AllReduce provides bandwidth-optimal gradient sync
3. **Gradient Bucketing**: Overlaps communication with backward pass
4. **DistributedSampler**: Ensures each GPU processes different data
5. **Linear Scaling**: Scale learning rate with batch size

### When to Use DDP

| Scenario | Recommendation |
|----------|----------------|
| Model fits in single GPU | Use DDP |
| Need maximum throughput | Use DDP |
| Simple setup required | Use DDP |
| Model exceeds GPU memory | Consider FSDP/ZeRO |

### Next Steps
- Learn FSDP for memory-efficient training
- Explore ZeRO optimization stages
- Combine with mixed precision (AMP)